# Delta Lake MERGE Operations

This notebook demonstrates Delta Lake's MERGE capabilities for upserts. We'll:

1. Generate a dataset with both new and updated records
2. Implement a MERGE statement matching on Order ID
3. Include conditional logic for updates vs inserts
4. Handle schema differences between source and target
5. Log the counts of inserted, updated, and unchanged records
6. Demonstrate transaction atomicity
7. Compare performance with alternative approaches
8. Show the impact of partitioning on MERGE operations

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, current_timestamp, rand
from delta.tables import DeltaTable

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
import utils

# Create Spark session with Delta Lake support
spark = SparkSession.builder \
    .appName("Delta Lake MERGE Operations") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 2. Load Delta Table and Get Initial Metrics

In [ ]:
# Define Delta table path
delta_table_path = "/opt/spark/data/processed/global_superstore_delta"

# Load the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# Get initial table metrics
initial_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
initial_version = initial_metrics["current_version"]

print(f"Initial table version: {initial_version}")
print(f"Initial record count: {initial_metrics['record_count']}")

## 3. Generate Source Data for MERGE

In [ ]:
# Read a sample of existing data to use as a template
existing_df = spark.read.format("delta").load(delta_table_path).limit(1000)
existing_count = existing_df.count()
print(f"Loaded {existing_count} existing records as template")

# Create a DataFrame with records to update (50% of sample)
records_to_update = existing_df.sample(fraction=0.5, seed=42)
update_count = records_to_update.count()
print(f"Selected {update_count} records to update")

# Modify the records to update
records_to_update = records_to_update \
    .withColumn("Sales", col("Sales") * (rand() * 0.4 + 0.8))  # Adjust sales by -20% to +20%
    .withColumn("Profit", col("Profit") * (rand() * 0.4 + 0.8))  # Adjust profit by -20% to +20%
    .withColumn("Quantity", (col("Quantity") * (rand() * 0.4 + 0.8)).cast("integer"))  # Adjust quantity
    .withColumn("Update_Source", lit("merge_update"))
    .withColumn("Last_Updated", current_timestamp())

# Create new records (not in the existing data)
# First, get a template record
template_df = existing_df.limit(100)

# Modify to create new records with unique Order IDs
new_records = template_df \
    .withColumn("Order ID", expr("concat('NEW-', uuid())"))  # Generate unique Order IDs
    .withColumn("Sales", col("Sales") * (rand() * 1.0 + 0.5))  # Random sales values
    .withColumn("Profit", col("Profit") * (rand() * 1.0 + 0.5))  # Random profit values
    .withColumn("Quantity", (rand() * 10 + 1).cast("integer"))  # Random quantity
    .withColumn("Update_Source", lit("merge_insert"))
    .withColumn("Last_Updated", current_timestamp())

new_count = new_records.count()
print(f"Generated {new_count} new records")

# Combine updates and inserts into a single source DataFrame
source_df = records_to_update.union(new_records)
source_count = source_df.count()
print(f"Combined source data has {source_count} records ({update_count} updates, {new_count} inserts)")

# Show sample of source data
source_df.select("Order ID", "Sales", "Profit", "Quantity", "Update_Source").show(5)

## 4. Add a New Column to Source Data (Schema Evolution)

In [ ]:
# Add a new column to the source data that doesn't exist in the target
source_df = source_df.withColumn("Customer_Satisfaction", (rand() * 5).cast("integer"))

# Show the updated schema
print("Source data schema with new column:")
source_df.printSchema()

# Show sample with new column
source_df.select("Order ID", "Sales", "Profit", "Customer_Satisfaction", "Update_Source").show(5)

## 5. Perform MERGE Operation

In [ ]:
# Start timing
merge_start_time = time.time()

# Perform MERGE operation
merge_result = delta_table.alias("target") \
    .merge(
        source_df.alias("source"),
        "target.`Order ID` = source.`Order ID`"
    ) \
    .whenMatchedUpdate(set={
        "Sales": "source.Sales",
        "Profit": "source.Profit",
        "Quantity": "source.Quantity",
        "Update_Source": "source.Update_Source",
        "Last_Updated": "source.Last_Updated",
        "Customer_Satisfaction": "source.Customer_Satisfaction"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

# End timing
merge_end_time = time.time()
merge_duration = merge_end_time - merge_start_time

print(f"MERGE operation completed in {merge_duration:.2f} seconds")

## 6. Verify MERGE Results

In [ ]:
# Get updated table metrics
updated_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
updated_version = updated_metrics["current_version"]
updated_count = updated_metrics["record_count"]

print(f"Table version changed from {initial_version} to {updated_version}")
print(f"Record count changed from {initial_metrics['record_count']} to {updated_count}")
print(f"Records added: {updated_count - initial_metrics['record_count']}")

# Check Delta table history
delta_table.history(1).show(truncate=False)

# Count records by update source
spark.read.format("delta").load(delta_table_path) \
    .groupBy("Update_Source") \
    .count() \
    .show()

## 7. Verify Schema Evolution

In [ ]:
# Check if the new column was added to the Delta table
updated_df = spark.read.format("delta").load(delta_table_path)
print("Updated Delta table schema:")
updated_df.printSchema()

# Count records with the new column populated
satisfaction_count = updated_df.filter(col("Customer_Satisfaction").isNotNull()).count()
print(f"Records with Customer_Satisfaction populated: {satisfaction_count}")

# Show sample records with the new column
updated_df.select("Order ID", "Sales", "Profit", "Customer_Satisfaction", "Update_Source") \
    .filter(col("Customer_Satisfaction").isNotNull()) \
    .show(5)

## 8. Compare with Alternative Approach (Separate INSERT/UPDATE)

In [ ]:
# Create a copy of the Delta table for comparison
comparison_table_path = "/opt/spark/data/processed/comparison_delta"
spark.read.format("delta").load(delta_table_path) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(comparison_table_path)

comparison_table = DeltaTable.forPath(spark, comparison_table_path)
print(f"Created comparison table at {comparison_table_path}")

# Split source data into updates and inserts
updates_df = source_df.filter(col("Update_Source") == "merge_update")
inserts_df = source_df.filter(col("Update_Source") == "merge_insert")

# Start timing for separate operations
separate_start_time = time.time()

# Perform UPDATE operation
update_start_time = time.time()
comparison_table.alias("target") \
    .update(
        condition=expr("target.`Order ID` IN (SELECT `Order ID` FROM updates)"),
        set={
            "Sales": "updates.Sales",
            "Profit": "updates.Profit",
            "Quantity": "updates.Quantity",
            "Update_Source": "updates.Update_Source",
            "Last_Updated": "updates.Last_Updated",
            "Customer_Satisfaction": "updates.Customer_Satisfaction"
        }
    )
update_end_time = time.time()
update_duration = update_end_time - update_start_time
print(f"UPDATE operation completed in {update_duration:.2f} seconds")

# Perform INSERT operation
insert_start_time = time.time()
inserts_df.write \
    .format("delta") \
    .mode("append") \
    .save(comparison_table_path)
insert_end_time = time.time()
insert_duration = insert_end_time - insert_start_time
print(f"INSERT operation completed in {insert_duration:.2f} seconds")

# End timing for separate operations
separate_end_time = time.time()
separate_duration = separate_end_time - separate_start_time

print(f"\nPerformance Comparison:")
print(f"MERGE operation: {merge_duration:.2f} seconds")
print(f"Separate UPDATE + INSERT: {separate_duration:.2f} seconds")
print(f"Difference: {separate_duration - merge_duration:.2f} seconds")
print(f"MERGE is {(separate_duration / merge_duration):.2f}x faster")

## 9. Demonstrate Conditional MERGE Logic

In [ ]:
# Generate source data with different conditions
conditional_source = source_df \
    .withColumn("Sales_Change", expr("Sales / 100"))  # Calculate sales change for conditions

# Perform conditional MERGE operation
print("Performing conditional MERGE operation...")
conditional_start_time = time.time()

delta_table.alias("target") \
    .merge(
        conditional_source.alias("source"),
        "target.`Order ID` = source.`Order ID`"
    ) \
    .whenMatchedUpdate(
        condition="source.Sales > target.Sales",  # Only update if sales increased
        set={
            "Sales": "source.Sales",
            "Profit": "source.Profit",
            "Quantity": "source.Quantity",
            "Update_Source": "'conditional_update_higher'",
            "Last_Updated": "current_timestamp()"
        }
    ) \
    .whenMatchedUpdate(
        condition="source.Sales <= target.Sales",  # Different update for decreased sales
        set={
            "Update_Source": "'conditional_update_lower'",
            "Last_Updated": "current_timestamp()"
        }
    ) \
    .whenNotMatchedInsert(
        condition="source.Sales > 100",  # Only insert if sales > 100
        values={
            "Row ID": "source.`Row ID`",
            "Order ID": "source.`Order ID`",
            "Order Date": "source.`Order Date`",
            "Ship Date": "source.`Ship Date`",
            "Ship Mode": "source.`Ship Mode`",
            "Customer ID": "source.`Customer ID`",
            "Customer Name": "source.`Customer Name`",
            "Segment": "source.Segment",
            "City": "source.City",
            "State": "source.State",
            "Country": "source.Country",
            "Postal Code": "source.`Postal Code`",
            "Market": "source.Market",
            "Region": "source.Region",
            "Product ID": "source.`Product ID`",
            "Category": "source.Category",
            "Sub-Category": "source.`Sub-Category`",
            "Product Name": "source.`Product Name`",
            "Sales": "source.Sales",
            "Quantity": "source.Quantity",
            "Discount": "source.Discount",
            "Profit": "source.Profit",
            "Shipping Cost": "source.`Shipping Cost`",
            "Order Priority": "source.`Order Priority`",
            "Update_Source": "'conditional_insert'",
            "Last_Updated": "current_timestamp()",
            "Customer_Satisfaction": "source.Customer_Satisfaction"
        }
    ) \
    .execute()

conditional_end_time = time.time()
conditional_duration = conditional_end_time - conditional_start_time
print(f"Conditional MERGE completed in {conditional_duration:.2f} seconds")

# Count records by update source to see the effect of conditions
spark.read.format("delta").load(delta_table_path) \
    .groupBy("Update_Source") \
    .count() \
    .orderBy("Update_Source") \
    .show(truncate=False)

## 10. Demonstrate Impact of Partitioning on MERGE

In [ ]:
# Create a partitioned and non-partitioned version of the table for comparison
base_df = spark.read.format("delta").load(delta_table_path)

# Create non-partitioned table
non_partitioned_path = "/opt/spark/data/processed/non_partitioned_delta"
base_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(non_partitioned_path)

# Create heavily partitioned table
partitioned_path = "/opt/spark/data/processed/partitioned_delta"
base_df.write \
    .format("delta") \
    .partitionBy("Ship Mode", "Category", "Segment") \
    .mode("overwrite") \
    .save(partitioned_path)

# Load tables
non_partitioned_table = DeltaTable.forPath(spark, non_partitioned_path)
partitioned_table = DeltaTable.forPath(spark, partitioned_path)

print("Created comparison tables for partitioning impact:")
print(f"Non-partitioned table: {non_partitioned_path}")
print(f"Partitioned table: {partitioned_path}")

# Generate test data that targets specific partitions
# We'll create updates that target a specific Ship Mode and Category
targeted_updates = base_df \
    .filter((col("Ship Mode") == "First Class") & (col("Category") == "Furniture")) \
    .withColumn("Sales", col("Sales") * 1.1) \
    .withColumn("Profit", col("Profit") * 1.1) \
    .withColumn("Update_Source", lit("partition_test"))

targeted_count = targeted_updates.count()
print(f"Generated {targeted_count} targeted updates for partition test")

# Test MERGE on non-partitioned table
print("\nPerforming MERGE on non-partitioned table...")
non_part_start = time.time()

non_partitioned_table.alias("target") \
    .merge(
        targeted_updates.alias("source"),
        "target.`Order ID` = source.`Order ID`"
    ) \
    .whenMatchedUpdate(set={
        "Sales": "source.Sales",
        "Profit": "source.Profit",
        "Update_Source": "source.Update_Source"
    }) \
    .execute()

non_part_end = time.time()
non_part_duration = non_part_end - non_part_start
print(f"MERGE on non-partitioned table completed in {non_part_duration:.2f} seconds")

# Test MERGE on partitioned table
print("\nPerforming MERGE on partitioned table...")
part_start = time.time()

partitioned_table.alias("target") \
    .merge(
        targeted_updates.alias("source"),
        "target.`Order ID` = source.`Order ID`"
    ) \
    .whenMatchedUpdate(set={
        "Sales": "source.Sales",
        "Profit": "source.Profit",
        "Update_Source": "source.Update_Source"
    }) \
    .execute()

part_end = time.time()
part_duration = part_end - part_start
print(f"MERGE on partitioned table completed in {part_duration:.2f} seconds")

# Compare performance
print(f"\nPartitioning Impact on MERGE Performance:")
print(f"Non-partitioned: {non_part_duration:.2f} seconds")
print(f"Partitioned: {part_duration:.2f} seconds")
print(f"Difference: {non_part_duration - part_duration:.2f} seconds")
if part_duration < non_part_duration:
    print(f"Partitioned table is {(non_part_duration / part_duration):.2f}x faster")
else:
    print(f"Non-partitioned table is {(part_duration / non_part_duration):.2f}x faster")

## 11. Summary

In this notebook, we've demonstrated Delta Lake's MERGE capabilities:

1. Performing atomic upserts with a single MERGE operation
2. Handling schema evolution during MERGE
3. Implementing conditional logic for updates and inserts
4. Comparing MERGE performance with separate INSERT/UPDATE operations
5. Analyzing the impact of partitioning on MERGE performance

MERGE operations are a powerful feature of Delta Lake that enable complex data integration patterns with atomicity guarantees. They are particularly useful for handling upserts, slowly changing dimensions, and other data integration scenarios.